# Tutorial: Averaged Converter Modes (CCM, DCM, Auto) in Pulsim

**Audience**
- Desenvolvedores e usuários do Pulsim que querem usar o novo `averaged_converter` com mais topologias e modos.

**Prerequisites**
- Noções básicas de conversores buck/boost/buck-boost.
- Python básico.

**Learning goals**
- Configurar `topology` e `operating_mode` no backend.
- Entender quando usar `ccm`, `dcm` e `auto`.
- Ler os canais canônicos `Iavg(...)`, `Vavg(...)`, `Davg`.
- Carregar a mesma configuração usando YAML.


## Outline

1. Setup do ambiente local
2. Helpers de simulação averaged
3. Caso buck em CCM
4. Caso boost em CCM
5. Caso buck-boost em DCM
6. Caso auto (troca CCM/DCM por limiar)
7. Exemplo completo via YAML parser
8. Pitfalls, exercícios e próximos passos


In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd().resolve())
build_python = repo_root / "build" / "python"
if build_python.exists() and str(build_python) not in sys.path:
    sys.path.insert(0, str(build_python))

import pulsim as ps

print("Repo root:", repo_root)
print("Build path:", build_python)
print("Pulsim version:", ps.__version__)
print("Has AveragedOperatingMode:", hasattr(ps, "AveragedOperatingMode"))

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


Repo root: /Users/lgili/Documents/01 - Codes/01 - Github/PulsimCore
Build path: /Users/lgili/Documents/01 - Codes/01 - Github/PulsimCore/build/python
Pulsim version: 0.7.0
Has AveragedOperatingMode: True


## Step 1 - Helpers para montar e rodar averaged

Todos os casos deste tutorial usam o mesmo mapeamento canônico:
- `vin_source = "Vin"`
- `inductor = "L1"`
- `capacitor = "C1"`
- `load_resistor = "Rload"`
- `output_node = "out"`

Isso facilita reutilizar as mesmas rotinas para qualquer topologia suportada.


In [2]:
TOPOLOGY = {
    "buck": ps.AveragedConverterTopology.Buck,
    "boost": ps.AveragedConverterTopology.Boost,
    "buck_boost": ps.AveragedConverterTopology.BuckBoost,
}

MODE = {
    "ccm": ps.AveragedOperatingMode.CCM,
    "dcm": ps.AveragedOperatingMode.DCM,
    "auto": ps.AveragedOperatingMode.Auto,
}


def build_converter_circuit(vin=12.0, l_h=100e-6, c_f=220e-6, r_load=10.0):
    ckt = ps.Circuit()
    n_in = ckt.add_node("in")
    n_out = ckt.add_node("out")
    gnd = ckt.ground()

    ckt.add_voltage_source("Vin", n_in, gnd, vin)
    ckt.add_inductor("L1", n_in, n_out, l_h, 0.0)
    ckt.add_capacitor("C1", n_out, gnd, c_f, 0.0)
    ckt.add_resistor("Rload", n_out, gnd, r_load)
    return ckt


def run_averaged_case(
    *,
    topology: str,
    mode: str,
    duty: float,
    r_load: float,
    tstop: float = 1e-3,
    dt: float = 1e-6,
    f_sw: float = 100e3,
    ccm_threshold: float = 0.0,
    envelope: str = "warn",
):
    ckt = build_converter_circuit(r_load=r_load)
    opts = ps.SimulationOptions()
    opts.tstart = 0.0
    opts.tstop = tstop
    opts.dt = dt

    avg = opts.averaged_converter
    avg.enabled = True
    avg.topology = TOPOLOGY[topology]
    avg.operating_mode = MODE[mode]
    avg.envelope_policy = (
        ps.AveragedEnvelopePolicy.Warn
        if envelope.lower() == "warn"
        else ps.AveragedEnvelopePolicy.Strict
    )
    avg.vin_source = "Vin"
    avg.inductor = "L1"
    avg.capacitor = "C1"
    avg.load_resistor = "Rload"
    avg.output_node = "out"
    avg.duty = duty
    avg.duty_min = 0.0
    avg.duty_max = 0.95
    avg.switching_frequency_hz = f_sw
    avg.initial_inductor_current = 0.0
    avg.initial_output_voltage = 0.0
    avg.ccm_current_threshold = ccm_threshold

    result = ps.Simulator(ckt, opts).run_transient(ckt.initial_state())
    if not result.success:
        raise RuntimeError(f"{result.diagnostic.name}: {result.message}")

    v = result.virtual_channels["Vavg(out)"]
    i = result.virtual_channels["Iavg(L1)"]
    d = result.virtual_channels["Davg"]

    summary = {
        "topology": topology,
        "mode": mode,
        "vout_final": float(v[-1]),
        "il_final": float(i[-1]),
        "duty_final": float(d[-1]),
        "steps": len(result.time),
    }
    return result, summary


def plot_channels(result, title: str):
    t = result.time
    v = result.virtual_channels["Vavg(out)"]
    i = result.virtual_channels["Iavg(L1)"]
    d = result.virtual_channels["Davg"]

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
    ax[0].plot(t, v)
    ax[0].set_title("Vavg(out)")
    ax[0].set_xlabel("time [s]")
    ax[0].set_ylabel("V")

    ax[1].plot(t, i)
    ax[1].set_title("Iavg(L1)")
    ax[1].set_xlabel("time [s]")
    ax[1].set_ylabel("A")

    ax[2].plot(t, d)
    ax[2].set_title("Davg")
    ax[2].set_xlabel("time [s]")
    ax[2].set_ylabel("ratio")

    fig.suptitle(title)
    fig.tight_layout()


## Step 2 - Buck em CCM

Aqui usamos `topology="buck"` e `operating_mode="ccm"`.


In [3]:
buck_ccm_result, buck_ccm_summary = run_averaged_case(
    topology="buck",
    mode="ccm",
    duty=0.40,
    r_load=10.0,
    ccm_threshold=0.0,
    envelope="warn",
)

print(buck_ccm_summary)
plot_channels(buck_ccm_result, "Buck | CCM")


{'topology': 'buck', 'mode': 'ccm', 'vout_final': 1.2303508430961312, 'il_final': 2.6817417950890623, 'duty_final': 0.4, 'steps': 1001}


## Step 3 - Boost em CCM

Para boost, o mesmo mapeamento de componentes é reutilizado.
A diferença é apenas `topology="boost"`.


In [4]:
boost_ccm_result, boost_ccm_summary = run_averaged_case(
    topology="boost",
    mode="ccm",
    duty=0.45,
    r_load=20.0,
    ccm_threshold=0.0,
    envelope="warn",
)

print(boost_ccm_summary)
plot_channels(boost_ccm_result, "Boost | CCM")


{'topology': 'boost', 'mode': 'ccm', 'vout_final': 38.69962585463331, 'il_final': -12.065538563916487, 'duty_final': 0.45, 'steps': 1001}


## Step 4 - Buck-Boost em DCM

No buck-boost ideal inverting, a saída média tende a ser negativa.


In [5]:
bb_dcm_result, bb_dcm_summary = run_averaged_case(
    topology="buck_boost",
    mode="dcm",
    duty=0.35,
    r_load=12.0,
    f_sw=100e3,
    ccm_threshold=5.0,  # Em DCM esse limiar não dispara falha de envelope CCM.
    envelope="strict",
)

print(bb_dcm_summary)
plot_channels(bb_dcm_result, "Buck-Boost | DCM")


{'topology': 'buck_boost', 'mode': 'dcm', 'vout_final': -1.0174880315942239, 'il_final': 0.07349999999999991, 'duty_final': 0.35, 'steps': 1001}


## Step 5 - Modo `auto`

`auto` escolhe o comportamento usando o limiar `ccm_current_threshold`.
A ideia é facilitar variação de carga sem trocar manualmente de modo.


In [6]:
auto_result, auto_summary = run_averaged_case(
    topology="boost",
    mode="auto",
    duty=0.45,
    r_load=60.0,
    ccm_threshold=2.0,
    envelope="strict",
)

forced_dcm_result, forced_dcm_summary = run_averaged_case(
    topology="boost",
    mode="dcm",
    duty=0.45,
    r_load=60.0,
    ccm_threshold=2.0,
    envelope="strict",
)

print("auto:", auto_summary)
print("forced_dcm:", forced_dcm_summary)
plot_channels(auto_result, "Boost | AUTO")


auto: {'topology': 'boost', 'mode': 'auto', 'vout_final': 1.2365056995147292, 'il_final': 0.40670259177452117, 'duty_final': 0.45, 'steps': 1001}
forced_dcm: {'topology': 'boost', 'mode': 'dcm', 'vout_final': 1.2365056995147292, 'il_final': 0.40670259177452117, 'duty_final': 0.45, 'steps': 1001}


## Step 6 - Mesmo setup via YAML

Os novos campos no YAML são:
- `topology`
- `operating_mode`
- `switching_frequency_hz`


In [7]:
yaml_text = '''
schema: pulsim-v1
version: 1
simulation:
  tstart: 0.0
  tstop: 8e-4
  dt: 1e-6
  averaged_converter:
    enabled: true
    topology: boost
    operating_mode: auto
    envelope_policy: warn
    vin_source: Vin
    inductor: L1
    capacitor: C1
    load_resistor: Rload
    output_node: out
    duty: 0.45
    duty_min: 0.0
    duty_max: 0.95
    switching_frequency_hz: 100000.0
    initial_inductor_current: 0.0
    initial_output_voltage: 0.0
    ccm_current_threshold: 1.0
components:
  - type: voltage_source
    name: Vin
    nodes: [in, 0]
    waveform: {type: dc, value: 12.0}
  - type: inductor
    name: L1
    nodes: [in, out]
    value: 100u
  - type: capacitor
    name: C1
    nodes: [out, 0]
    value: 220u
  - type: resistor
    name: Rload
    nodes: [out, 0]
    value: 20
'''

parser = ps.YamlParser()
circuit, options = parser.load_string(yaml_text)

print("Parser errors:", parser.errors)
print("Topologia:", options.averaged_converter.topology)
print("Modo:", options.averaged_converter.operating_mode)
print("f_sw:", options.averaged_converter.switching_frequency_hz)

yaml_result = ps.Simulator(circuit, options).run_transient(circuit.initial_state())
print("success:", yaml_result.success, "diag:", yaml_result.diagnostic.name)
print("Vavg(out) final:", float(yaml_result.virtual_channels["Vavg(out)"][-1]))


Parser errors: []
Topologia: AveragedConverterTopology.Boost
Modo: AveragedOperatingMode.Auto
f_sw: 100000.0
success: True diag: None
Vavg(out) final: 2.312974927349619


## Pitfalls comuns

- **Esquecer `switching_frequency_hz`**: em `dcm/auto` o backend precisa desse valor.
- **Mapeamento de componentes errado**: `vin_source`, `inductor`, `capacitor`, `load_resistor` devem apontar para tipos corretos.
- **Interpretar `Davg` como sinal PWM lógico**: `Davg` é razão cíclica média, não estado 0/1.

## Exercise

1. Troque a carga do caso `boost|auto` de `60.0` para `15.0`.
2. Compare o `Vavg(out)` final em `auto`, `ccm` e `dcm`.
3. Ajuste `ccm_current_threshold` e observe quando o comportamento muda.


In [8]:
# Exercise scaffold
# TODO: altere os parâmetros e compare os resultados.

cases = [
    ("auto", 15.0),
    ("ccm", 15.0),
    ("dcm", 15.0),
]

for mode, load in cases:
    result, summary = run_averaged_case(
        topology="boost",
        mode=mode,
        duty=0.45,
        r_load=load,
        ccm_threshold=1.0,
        envelope="warn",
    )
    print(mode, summary["vout_final"], summary["il_final"])


auto 40.100596630911795 0.9841874186037828
ccm 38.186948265397156 -10.352293101477127
dcm 3.527163504404038 1.0286488933242373
